In [1]:
# Load .env (ORA_USER, ORA_PASSWORD, ORA_DSN, PDB_NAME) and check the values exist
import os
import json
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
import oracledb

# project root = parent of notebook/, where .env lives
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

# read the four required variables
ORA_USER = os.getenv("ORA_USER")
ORA_PASSWORD = os.getenv("ORA_PASSWORD")
ORA_DSN = os.getenv("ORA_DSN")
PDB_NAME = os.getenv("PDB_NAME")

# fail fast with a clear message if something is missing
missing = [n for n, v in {"ORA_USER": ORA_USER, "ORA_PASSWORD": ORA_PASSWORD,
                          "ORA_DSN": ORA_DSN, "PDB_NAME": PDB_NAME}.items() if not v]
if missing:
    raise ValueError(f"Missing variables in .env: {', '.join(missing)}")

print(f"PDB: {PDB_NAME} | credentials loaded (not shown)")

PDB: text2sql | credentials loaded (not shown)


In [2]:
# Output goes to metadata/<PDB_NAME>/ at the project root
OUT_DIR = PROJECT_ROOT / "metadata" / PDB_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"output dir: {OUT_DIR}")

output dir: /workspace/project/queryllm/metadata/text2sql


In [3]:
# Verify the credentials work, then close immediately
# thin mode = no Oracle Client installation needed
with oracledb.connect(user=ORA_USER, password=ORA_PASSWORD, dsn=ORA_DSN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT USER FROM dual")
        db_user = cur.fetchone()[0]
print(f"connection ok as {db_user}")

connection ok as FDR_AGENT


In [4]:
# One query -> one cursor -> list of rows
# every extraction function below uses this, no global cursors
def fetch_all(conn, sql, **binds):
    with conn.cursor() as cur:
        cur.execute(sql, binds)
        return cur.fetchall()

In [5]:
# Find user schemas that own at least one TABLE / VIEW / MATERIALIZED VIEW
# ALL_USERS.ORACLE_MAINTAINED='N' skips pseudo-users (SYS, SYSTEM, ...)
# owner <> USER skips the extraction user's own internal objects
def extract_schemas(conn):
    rows = fetch_all(
        conn,
        """
        SELECT DISTINCT o.owner
        FROM all_objects o
        JOIN all_users u ON u.username = o.owner
        WHERE u.oracle_maintained = 'N'
          AND o.object_type IN ('TABLE', 'VIEW', 'MATERIALIZED VIEW')
          AND o.owner <> USER
        ORDER BY o.owner
        """,
    )
    return [row[0] for row in rows]

In [6]:
# One record per TABLE / VIEW / MATERIALIZED VIEW in a schema
# a materialized view also shows up as a TABLE row in ALL_OBJECTS,
# so ALL_MVIEWS is authoritative: matching names are marked MV and the
# duplicate TABLE row is collapsed instead of overwriting the MV record
def extract_objects(conn, schema):
    mview_names = {
        row[0].upper()
        for row in fetch_all(conn, "SELECT mview_name FROM all_mviews WHERE owner = :owner", owner=schema)
    }
    rows = fetch_all(
        conn,
        """
        SELECT DISTINCT o.object_name, o.object_type, c.comments
        FROM all_objects o
        LEFT JOIN all_tab_comments c
          ON c.owner = o.owner AND c.table_name = o.object_name
        WHERE o.owner = :owner
          AND o.object_type IN ('TABLE', 'VIEW', 'MATERIALIZED VIEW')
        ORDER BY o.object_name
        """,
        owner=schema,
    )
    objects = {}
    for name, object_type, comment in rows:
        if name in objects:
            continue  # duplicate row (MV container table) already recorded
        is_mv = name.upper() in mview_names
        objects[name] = {
            "object_type": "MATERIALIZED VIEW" if is_mv else object_type,
            "description": comment or "",
            "columns": [],
            "primary_key": [],
            "primary_key_name": None,
            "unique_constraints": [],
            "foreign_keys": [],
        }
    return objects

In [7]:
# Fill objects[name]["columns"] in place
# NUMBER reports DATA_LENGTH=22 which is meaningless as a size,
# so DATA_PRECISION / DATA_SCALE are read instead
def extract_columns(conn, schema, objects):
    rows = fetch_all(
        conn,
        """
        SELECT tc.table_name, tc.column_name, tc.data_type,
               tc.data_length, tc.data_precision, tc.data_scale,
               tc.nullable, tc.column_id, cc.comments
        FROM all_tab_columns tc
        LEFT JOIN all_col_comments cc
          ON cc.owner = tc.owner
         AND cc.table_name = tc.table_name
         AND cc.column_name = tc.column_name
        WHERE tc.owner = :owner
        ORDER BY tc.table_name, tc.column_id
        """,
        owner=schema,
    )
    for tname, cname, dtype, dlength, precision, scale, nullable, col_id, comment in rows:
        if tname not in objects:
            continue
        objects[tname]["columns"].append({
            "name": cname,
            "type": dtype,
            "length": dlength,
            "precision": precision,
            "scale": scale,
            "nullable": nullable == "Y",
            "position": col_id,
            "description": comment or "",
        })

In [8]:
# Primary keys, unique constraints and foreign keys
# FKs are resolved through (R_OWNER, R_CONSTRAINT_NAME), not the constraint
# name alone, so cross-schema relationships resolve correctly; if the
# referenced key is invisible to the extraction user the FK is kept with
# reference_resolved = false
def extract_keys(conn, schema, objects):
    # group every P/U key by (owner, constraint_name) across all visible schemas
    key_map = {}
    key_rows = fetch_all(
        conn,
        """
        SELECT c.owner, c.constraint_name, c.constraint_type,
               c.table_name, c.status, c.validated, cc.column_name
        FROM all_constraints c
        JOIN all_cons_columns cc
          ON cc.owner = c.owner
         AND cc.constraint_name = c.constraint_name
         AND cc.table_name = c.table_name
        WHERE c.constraint_type IN ('P', 'U')
        ORDER BY c.owner, c.constraint_name, cc.position
        """,
    )
    for key_owner, cname, ctype, tname, status, validated, col in key_rows:
        key = key_map.setdefault(
            (key_owner, cname),
            {"table": tname, "type": ctype, "status": status, "validated": validated, "columns": []},
        )
        key["columns"].append(col)

    # attach PKs and unique keys to this schema's objects
    for (key_owner, cname), key in key_map.items():
        if key_owner != schema or key["table"] not in objects:
            continue
        if key["type"] == "P":
            objects[key["table"]]["primary_key"] = list(key["columns"])
            objects[key["table"]]["primary_key_name"] = cname
        else:
            objects[key["table"]]["unique_constraints"].append({
                "name": cname,
                "columns": list(key["columns"]),
            })

    # foreign keys of this schema, resolved via the (r_owner, r_cname) pair
    fk_rows = fetch_all(
        conn,
        """
        SELECT c.table_name, c.constraint_name, cc.column_name,
               c.r_owner, c.r_constraint_name
        FROM all_constraints c
        JOIN all_cons_columns cc
          ON cc.owner = c.owner
         AND cc.constraint_name = c.constraint_name
         AND cc.table_name = c.table_name
        WHERE c.owner = :owner
          AND c.constraint_type = 'R'
        ORDER BY c.table_name, c.constraint_name, cc.position
        """,
        owner=schema,
    )
    fks = {}
    for tname, cname, col, r_owner, r_cname in fk_rows:
        if tname not in objects:
            continue
        if (tname, cname) not in fks:
            ref = key_map.get((r_owner, r_cname))
            fk = {
                "name": cname,
                "columns": [],
                "references_schema": r_owner,
                "references_table": ref["table"] if ref else None,
                "references_columns": list(ref["columns"]) if ref else [],
                "reference_resolved": ref is not None,
            }
            fks[(tname, cname)] = fk
            objects[tname]["foreign_keys"].append(fk)
        fks[(tname, cname)]["columns"].append(col)

In [9]:
# Run the extraction for every schema and write one JSON file per schema
# plus a manifest.json summarising the run; the connection is closed in
# finally even if something fails midway
def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

extracted_at = datetime.now(timezone.utc).isoformat()
all_metadata = {}

conn = oracledb.connect(user=ORA_USER, password=ORA_PASSWORD, dsn=ORA_DSN)
try:
    schemas = extract_schemas(conn)
    print(f"schemas: {schemas}")

    for schema in schemas:
        objects = extract_objects(conn, schema)
        extract_columns(conn, schema, objects)
        extract_keys(conn, schema, objects)

        all_metadata[schema] = {
            "metadata_version": "1.0",
            "database": PDB_NAME,
            "schema": schema,
            "extracted_at": extracted_at,
            "objects": objects,
        }
        print(f"{schema}: {len(objects)} objects")

    for schema, payload in all_metadata.items():
        write_json(OUT_DIR / f"{schema}.json", payload)

    manifest = {
        "metadata_version": "1.0",
        "database": PDB_NAME,
        "extracted_at": extracted_at,
        "schemas": {
            s: {"object_count": len(p["objects"]), "file": f"{s}.json"}
            for s, p in all_metadata.items()
        },
    }
    write_json(OUT_DIR / "manifest.json", manifest)
    print("manifest written")
finally:
    conn.close()
    print("connection closed")

schemas: ['CO', 'HR', 'SH']


CO: 11 objects
HR: 8 objects
SH: 12 objects
manifest written
connection closed


In [10]:
# Quick human-readable summary of what was extracted
total = 0
for schema in sorted(all_metadata):
    payload = all_metadata[schema]
    counts = {}
    for obj in payload["objects"].values():
        counts[obj["object_type"]] = counts.get(obj["object_type"], 0) + 1
    total += len(payload["objects"])
    print(f"{schema} ({len(payload['objects'])} objects): {counts}")
print(f"total: {total} objects in {len(all_metadata)} schemas -> {OUT_DIR}")

CO (11 objects): {'TABLE': 7, 'VIEW': 4}
HR (8 objects): {'TABLE': 7, 'VIEW': 1}
SH (12 objects): {'MATERIALIZED VIEW': 2, 'TABLE': 9, 'VIEW': 1}
total: 31 objects in 3 schemas -> /workspace/project/queryllm/metadata/text2sql
